In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os


In [2]:
instamart=pd.read_csv("C:/Users/HP/Downloads/instamart/order_products__prior.csv")
orders = pd.read_csv("C:/Users/HP/Downloads/instamart/orders.csv")
products = pd.read_csv("C:/Users/HP/Downloads/instamart/products.csv")
aisles = pd.read_csv("C:/Users/HP/Downloads/instamart/aisles.csv")
departments = pd.read_csv("C:/Users/HP/Downloads/instamart/departments.csv")




In [3]:
instamart.head()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [4]:
#Compare reorder rates between morning orders and evening orders.
merge=instamart.merge(orders,on='order_id')

def time_period(hour):
    if hour <12:
        return 'Morning'
    elif hour >=18:
        return 'Evening'
    return 'other'

merge['time_period']=merge['order_hour_of_day'].apply(time_period)
merge[merge['time_period'].isin(['Morning','Evening'])].groupby('time_period')['reordered'].mean()*100

time_period
Evening    57.928157
Morning    60.876496
Name: reordered, dtype: float64

In [5]:
# What is the most popular product ordered on Saturdays specifically?
merged = instamart.merge(orders, on='order_id').merge(products, on='product_id')
saturday_orders = merged[merged['order_dow'] == 0]  # adjust encoding
saturday_orders['product_name'].value_counts().idxmax()

'Banana'

In [6]:
orders.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


In [7]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 3421083 entries, 0 to 3421082
Data columns (total 7 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   user_id                 int64  
 2   eval_set                str    
 3   order_number            int64  
 4   order_dow               int64  
 5   order_hour_of_day       int64  
 6   days_since_prior_order  float64
dtypes: float64(1), int64(5), str(1)
memory usage: 182.7 MB


In [8]:
user_order_count=orders['user_id'].value_counts()


In [9]:
# how many user order place most and how many user 
max_orders=user_order_count.max()
max_orders
num_user_with_max=(user_order_count==max_orders).sum()
num_user_with_max

np.int64(1374)

In [10]:
#what percentage of orders happen on weekend
weekend_order=orders[orders['order_dow'].isin([0,6])].shape[0]
percentage=(weekend_order/orders.shape[0])*100
percentage

30.682272251214016

In [11]:
#What is the most common order_hour_of_day? 
orders['order_hour_of_day'].value_counts().idxmax()

np.int64(10)

In [12]:
#  How many orders have order_number = 1 (first-time orders)?
(orders['order_number']==1).sum()


np.int64(206209)

In [13]:
#What is the busiest hour on Saturdays specifically?
saturday_order=orders.query('order_dow == 0')
saturday_order['order_hour_of_day'].value_counts().idxmax()

np.int64(14)

In [14]:
departments.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   department_id  21 non-null     int64
 1   department     21 non-null     str  
dtypes: int64(1), str(1)
memory usage: 468.0 bytes


In [15]:
departments.duplicated().sum()

np.int64(0)

aisles

In [16]:
aisles.head()

,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


In [17]:
aisles.info()

<class 'pandas.DataFrame'>
RangeIndex: 134 entries, 0 to 133
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   aisle_id  134 non-null    int64
 1   aisle     134 non-null    str  
dtypes: int64(1), str(1)
memory usage: 2.2 KB


In [18]:
aisles.isnull().sum()

aisle_id    0
aisle       0
dtype: int64

In [19]:
aisles.duplicated().sum()

np.int64(0)

merge table (order_details)

In [20]:
check_order=pd.merge(orders,instamart,on='order_id',how='inner')
check_order

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered
0,2539329,1,prior,1,2,8,NaN,196,1,0
1,2539329,1,prior,1,2,8,NaN,14084,2,0
2,2539329,1,prior,1,2,8,NaN,12427,3,0
3,2539329,1,prior,1,2,8,NaN,26088,4,0
4,2539329,1,prior,1,2,8,NaN,26405,5,0
...,...,...,...,...,...,...,...,...,...,...
32434484,2977660,206209,prior,13,1,12,7.0,14197,5,1
32434485,2977660,206209,prior,13,1,12,7.0,38730,6,0
32434486,2977660,206209,prior,13,1,12,7.0,31477,7,0
32434487,2977660,206209,prior,13,1,12,7.0,6567,8,0


In [21]:
order_details=pd.merge(check_order,products,on='product_id')
order_details

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id
0,2539329,1,prior,1,2,8,NaN,196,1,0,Soda,77,7
1,2539329,1,prior,1,2,8,NaN,14084,2,0,Organic Unsweetened Vanilla Almond Milk,91,16
2,2539329,1,prior,1,2,8,NaN,12427,3,0,Original Beef Jerky,23,19
3,2539329,1,prior,1,2,8,NaN,26088,4,0,Aged White Cheddar Popcorn,23,19
4,2539329,1,prior,1,2,8,NaN,26405,5,0,XL Pick-A-Size Paper Towel Rolls,54,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32434484,2977660,206209,prior,13,1,12,7.0,14197,5,1,Tomato Paste,9,9
32434485,2977660,206209,prior,13,1,12,7.0,38730,6,0,Brownie Crunch High Protein Bar,3,19
32434486,2977660,206209,prior,13,1,12,7.0,31477,7,0,High Protein Bar Chunky Peanut Butter,3,19
32434487,2977660,206209,prior,13,1,12,7.0,6567,8,0,Chocolate Peanut Butter Protein Bar,3,19


apriori

In [22]:
#products_per_order=order_details.groupby('order_id')['product_id'].count().reset_index(name='total_products')#get product count per order 
#products_per_order

In [23]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 3421083 entries, 0 to 3421082
Data columns (total 7 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   user_id                 int64  
 2   eval_set                str    
 3   order_number            int64  
 4   order_dow               int64  
 5   order_hour_of_day       int64  
 6   days_since_prior_order  float64
dtypes: float64(1), int64(5), str(1)
memory usage: 182.7 MB


In [24]:
products.head()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [25]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 49688 entries, 0 to 49687
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   product_id     49688 non-null  int64
 1   product_name   49688 non-null  str  
 2   aisle_id       49688 non-null  int64
 3   department_id  49688 non-null  int64
dtypes: int64(3), str(1)
memory usage: 1.5 MB


In [26]:
products.isnull().sum()

product_id       0
product_name     0
aisle_id         0
department_id    0
dtype: int64

In [27]:
instamart.tail()

,order_id,product_id,add_to_cart_order,reordered
32434484,3421083,39678,6,1
32434485,3421083,11352,7,0
32434486,3421083,4600,8,0
32434487,3421083,24852,9,1
32434488,3421083,5020,10,1


In [28]:
instamart.info()

<class 'pandas.DataFrame'>
RangeIndex: 32434489 entries, 0 to 32434488
Data columns (total 4 columns):
 #   Column             Dtype
---  ------             -----
 0   order_id           int64
 1   product_id         int64
 2   add_to_cart_order  int64
 3   reordered          int64
dtypes: int64(4)
memory usage: 989.8 MB


In [29]:
instamart.groupby('reordered')['order_id'].count()

reordered
0    13307953
1    19126536
Name: order_id, dtype: int64

In [30]:
instamart.isnull().sum()

order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64

In [31]:
aisles.info()
aisles['aisle_id']=aisles['aisle_id'].astype(int)

<class 'pandas.DataFrame'>
RangeIndex: 134 entries, 0 to 133
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   aisle_id  134 non-null    int64
 1   aisle     134 non-null    str  
dtypes: int64(1), str(1)
memory usage: 2.2 KB


In [32]:
products_full=products.merge(aisles,on='aisle_id').merge(departments,on='department_id')
products_full

,product_id,product_name,aisle_id,department_id,aisle,department
0,1,Chocolate Sandwich Cookies,61,19,cookies cakes,snacks
1,2,All-Seasons Salt,104,13,spices seasonings,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,tea,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,frozen meals,frozen
4,5,Green Chile Anytime Sauce,5,13,marinades meat preparation,pantry
...,...,...,...,...,...,...
49683,49684,"Vodka, Triple Distilled, Twist of Vanilla",124,5,spirits,alcohol
49684,49685,En Croute Roast Hazelnut Cranberry,42,1,frozen vegan vegetarian,frozen
49685,49686,Artisan Baguette,112,3,bread,bakery
49686,49687,Smartblend Healthy Metabolism Dry Cat Food,41,8,cat food care,pets


In [33]:
orders.groupby('order_dow')['order_id'].nunique().sort_values(ascending=False)# find which dow is place order high 

order_dow
0    600905
1    587478
2    467260
5    453368
6    448761
3    436972
4    426339
Name: order_id, dtype: int64

In [34]:
orders.groupby('order_hour_of_day')['order_id'].count().sort_values(ascending=False)#order place by order_hour_of_day

order_hour_of_day
10    288418
11    284728
15    283639
14    283042
13    277999
12    272841
16    272553
9     257812
17    228795
18    182912
8     178201
19    140569
20    104292
7      91868
21     78109
22     61468
23     40043
6      30529
0      22758
1      12398
5       9569
2       7539
4       5527
3       5474
Name: order_id, dtype: int64

In [35]:
first_five_depart=departments['department']

for dept in first_five_depart:
    count=products_full[products_full['department']==dept].shape[0]
    print (f"{dept}:{count} products")

frozen:4007 products
other:548 products
bakery:1516 products
produce:1684 products
alcohol:1054 products
international:1139 products
beverages:4365 products
pets:972 products
dry goods pasta:1858 products
bulk:38 products
personal care:6563 products
meat seafood:907 products
pantry:5371 products
breakfast:1115 products
canned goods:2092 products
dairy eggs:3449 products
household:3085 products
babies:1081 products
snacks:6264 products
deli:1322 products
missing:1258 products


practise

In [36]:
full_data = (
    instamart
    .merge(orders, on='order_id', how='inner')
    .merge(products_full, on='product_id', how='inner')
)

# Check the result
print(full_data.shape)   # expected: (32434489, 13)
print(full_data.columns.tolist())
full_data.head()

(32434489, 15)
['order_id', 'product_id', 'add_to_cart_order', 'reordered', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'product_name', 'aisle_id', 'department_id', 'aisle', 'department']


,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id,aisle,department
0,2,33120,1,1,202279,prior,3,5,9,8.0,Organic Egg Whites,86,16,eggs,dairy eggs
1,2,28985,2,1,202279,prior,3,5,9,8.0,Michigan Organic Kale,83,4,fresh vegetables,produce
2,2,9327,3,0,202279,prior,3,5,9,8.0,Garlic Powder,104,13,spices seasonings,pantry
3,2,45918,4,1,202279,prior,3,5,9,8.0,Coconut Butter,19,13,oils vinegars,pantry
4,2,30035,5,0,202279,prior,3,5,9,8.0,Natural Sweetener,17,13,baking ingredients,pantry


In [37]:
user_total=orders.groupby('user_id')['order_id'].nunique()
full_data['total_orders']=full_data['user_id'].map(user_total).astype('int16')

In [38]:
full_data.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id,aisle,department,total_orders
0,2,33120,1,1,202279,prior,3,5,9,8.0,Organic Egg Whites,86,16,eggs,dairy eggs,9
1,2,28985,2,1,202279,prior,3,5,9,8.0,Michigan Organic Kale,83,4,fresh vegetables,produce,9
2,2,9327,3,0,202279,prior,3,5,9,8.0,Garlic Powder,104,13,spices seasonings,pantry,9
3,2,45918,4,1,202279,prior,3,5,9,8.0,Coconut Butter,19,13,oils vinegars,pantry,9
4,2,30035,5,0,202279,prior,3,5,9,8.0,Natural Sweetener,17,13,baking ingredients,pantry,9
